In [23]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import pandas as pd

DATA_DIR = Path("/Users/temp/Documents/GitHub/MappingAffordances/Data_Input")
FIG_DIR = DATA_DIR / "preview"
RAW_GPKG = DATA_DIR / "nyc_parks_osm.gpkg"
ATTR_CSV = DATA_DIR / "nyc_parks_osm_attributes.csv"

PLACE = "New York City, New York, USA"
TAGS = {"leisure": "park"}

# Tags to keep as columns (missing ones are skipped)
KEEP_TAGS = [
    "name",
    "operator",
    "access",
    "fee",
    "owner",
    "ownership",
    "park:type",
    "wikidata",
    "website",
]

# `access` values meaning "not open to the general public"
RESTRICTED_ACCESS = {"private", "no", "customers", "permit", "members"}

# Servers are tried in order. Public mirrors come and go, so edit this list if needed.
OVERPASS_SERVERS = [
    "https://overpass.kumi.systems/api",
    "https://overpass.private.coffee/api",
    "https://overpass-api.de/api",
]

ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.requests_timeout = 180
ox.settings.timeout = 180
ox.settings.requests_headers = {
    "User-Agent": "MappingAffordances/0.1 (your-zacalewis@gmail.com)"  # <- put your email
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("OSMnx version:", ox.__version__)

OSMnx version: 2.1.1


In [24]:
import glob


def load_parks() -> gpd.GeoDataFrame:
    """Combine the per-borough GeoJSON files exported from Overpass Turbo."""
    files = sorted(glob.glob(str(DATA_DIR / "parks_*.geojson")))
    if not files:
        raise FileNotFoundError(f"No parks_*.geojson files found in {DATA_DIR}")
    print("Loading:", [Path(f).name for f in files])

    parts = []
    for f in files:
        g = gpd.read_file(f)
        g["source_file"] = Path(f).stem  # e.g. parks_brooklyn
        parts.append(g)
    gdf = pd.concat(parts, ignore_index=True)

    # Polygons only; a park that touches two boroughs can appear twice
    gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    idcol = "@id" if "@id" in gdf.columns else "id"
    gdf = gdf.drop_duplicates(subset=idcol)

    gdf["osm_type"] = gdf[idcol].astype(str).str.split("/").str[0]
    gdf["osm_id"] = gdf[idcol].astype(str).str.split("/").str[1]

    # Keep only the columns we want. Turbo adds list-type columns
    # (like @relations) that can't be written to a GeoPackage.
    cols = (
        ["osm_type", "osm_id", "source_file"]
        + [c for c in KEEP_TAGS if c in gdf.columns]
        + ["geometry"]
    )
    return gpd.GeoDataFrame(gdf[cols], geometry="geometry", crs="EPSG:4326")

## Graph Preview

In [25]:
COLORS = {
    "Open & free (or unmapped)": "#2e7d32",
    "Fee charged": "#ef6c00",
    "Restricted access": "#c62828",
}


def static_figures(gdf: gpd.GeoDataFrame) -> None:
    g = gdf.to_crs(3857)

    fig, ax = plt.subplots(figsize=(11, 11))
    for cat, color in COLORS.items():
        sub = g[g["category"] == cat]
        if len(sub):
            sub.plot(
                ax=ax,
                color=color,
                edgecolor="none",
                alpha=0.85,
                label=f"{cat} ({len(sub):,})",
            )
    try:
        import contextily as cx

        cx.add_basemap(ax, source=cx.providers.CartoDB.PositronNoLabels)
    except Exception:
        pass  # basemap is optional
    ax.set_axis_off()
    ax.legend(loc="upper left")
    ax.set_title("NYC parks in OpenStreetMap (leisure=park)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "parks_map.png", dpi=200)
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    comp = gdf[["has_operator", "has_access", "has_fee"]].mean() * 100
    comp.index = ["operator", "access", "fee"]
    comp.plot.bar(ax=axes[0], color="#455a64", rot=0)
    axes[0].set_ylabel("% of parks with tag")
    axes[0].set_ylim(0, 100)
    axes[0].set_title("Tag completeness")

    gdf["operator"].value_counts().head(10).sort_values().plot.barh(
        ax=axes[1], color="#455a64"
    )
    axes[1].set_title("Top operator values")

    gdf["area_acres"].clip(upper=gdf["area_acres"].quantile(0.99)).plot.hist(
        ax=axes[2], bins=40, color="#455a64"
    )
    axes[2].set_xlabel("acres (clipped at 99th percentile)")
    axes[2].set_title("Park size distribution")

    fig.tight_layout()
    fig.savefig(FIG_DIR / "parks_data_quality.png", dpi=150)
    plt.show()


def interactive_map(gdf: gpd.GeoDataFrame):
    """Zoomable map; click a park to see its tags. Also saved as HTML."""
    present = sorted(gdf["category"].unique())
    tooltip = [
        c
        for c in ["name", "operator", "access", "fee", "area_acres"]
        if c in gdf.columns
    ]
    m = gdf.explore(
        column="category",
        categorical=True,
        cmap=[COLORS[c] for c in present],
        tooltip=tooltip,
        tiles="CartoDB positron",
        style_kwds={"weight": 0.5, "fillOpacity": 0.7},
    )
    out = FIG_DIR / "parks_interactive.html"
    m.save(out)
    print(f"Saved interactive map -> {out}  (open in a browser if it won't draw here)")
    return m

In [26]:
import requests, time

for url in OVERPASS_SERVERS:
    t = time.time()
    try:
        r = requests.get(url + "/status", timeout=20)
        print(f"{url}  ->  {r.status_code}  in {time.time() - t:.1f}s")
    except Exception as e:
        print(f"{url}  ->  {type(e).__name__}")

https://overpass.kumi.systems/api  ->  429  in 0.4s
https://overpass.private.coffee/api  ->  429  in 0.5s
https://overpass-api.de/api  ->  406  in 0.4s


In [27]:
gdf = classify(load_parks())

gdf.to_file(RAW_GPKG, driver="GPKG", layer="parks")
gdf.drop(columns="geometry").to_csv(ATTR_CSV, index=False)
print(f"Saved {len(gdf):,} park polygons -> {RAW_GPKG}")

print("\nPolygons per file:")
print(gdf["source_file"].value_counts().to_string())
gdf.head()

FileNotFoundError: No parks_*.geojson files found in /Users/temp/Documents/GitHub/MappingAffordances/Data_Input

In [ ]:
summarize(gdf)

In [ ]:
static_figures(gdf)

In [ ]:
interactive_map(gdf)